In [6]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import random, shutil

# Mengunci random seed agar eksperimen konsisten dan dapat direproduksi (reproducible)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.1


In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".webp")

# =====================================================================
# <-- SESUAIKAN PATH FOLDER UTAMA DI BAWAH INI -->
# =====================================================================
BASE_PROJECT_DIR = "D:\SEMESTER 5\TATA TULIS ILMIAH\WEEK 12\image_classification_vokal_SIBI"
RAW_SOURCE_DIR   = os.path.join(BASE_PROJECT_DIR, "dataset")
CLASSES          = ['A', 'E', 'I', 'O', 'U']
# =====================================================================

counts = {}
basenames = []
corrupt_files = []

for cls in CLASSES:
    cls_dir = os.path.join(RAW_SOURCE_DIR, cls)
    files = [
        f for f in glob.glob(os.path.join(cls_dir, "*.*"))
        if f.lower().endswith(EXTS)
    ]
    valid_count = 0
    for f in files:
        img = cv2.imread(f)
        if img is None:
            corrupt_files.append(f)
        else:
            valid_count += 1
            basenames.append(os.path.basename(f))
    counts[cls] = valid_count
    print(f"Kelas {cls}: {valid_count} citra valid (total file terbaca: {len(files)})")

total_imgs = sum(counts.values())
print(f"\nTotal citra valid : {total_imgs}")
print(f"File korup/gagal  : {len(corrupt_files)}")
print(f"Nama file unik    : {len(set(basenames))} dari {len(basenames)} file")

if corrupt_files:
    print("Daftar file korup:", corrupt_files[:10])

In [ ]:
SPLIT_TRAIN_DIR = os.path.join(BASE_PROJECT_DIR, "split", "train")
TEST_DIR        = os.path.join(BASE_PROJECT_DIR, "split", "test")

CLASSES         = ['A', 'E', 'I', 'O', 'U']
TEST_RATIO      = 0.20
EXTS            = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
# =====================================================================

random.seed(SEED)

total_train, total_test = 0, 0

for cls in CLASSES:
    src_cls_dir   = os.path.join(RAW_SOURCE_DIR, cls)
    dst_train_cls = os.path.join(SPLIT_TRAIN_DIR, cls)
    dst_test_cls  = os.path.join(TEST_DIR, cls)

    os.makedirs(dst_train_cls, exist_ok=True)
    os.makedirs(dst_test_cls, exist_ok=True)

    files = [
        f for f in glob.glob(os.path.join(src_cls_dir, "*"))
        if os.path.isfile(f) and f.lower().endswith(EXTS)
    ]
    files.sort()
    random.shuffle(files)

    n_test      = max(1, int(round(len(files) * TEST_RATIO)))
    test_files  = files[:n_test]
    train_files = files[n_test:]

    for idx, f in enumerate(train_files):
        safe_name = f"{cls}_tr_{idx:04d}_{os.path.basename(f)}"
        shutil.copy2(f, os.path.join(dst_train_cls, safe_name))

    for idx, f in enumerate(test_files):
        safe_name = f"{cls}_te_{idx:04d}_{os.path.basename(f)}"
        shutil.copy2(f, os.path.join(dst_test_cls, safe_name))

    total_train += len(train_files)
    total_test  += len(test_files)
    print(f"Kelas {cls}: total={len(files)} -> train={len(train_files)}, test={len(test_files)}")

print(f"\nSelesai! Total train={total_train}, total test={total_test}")